In [2]:
!pip install boto3 opensearch-py -q

In [3]:
import os
import json
import boto3
from opensearchpy import OpenSearch, helpers

In [ ]:
# ==========================================
# 1. CREDENTIALS & CONFIGURATION
# ==========================================
# OpenSearch Credentials (Use the Master User you created in the AWS Console)
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST")
OPENSEARCH_USER = os.getenv("OPENSEARCH_USER")
OPENSEARCH_PASS = os.getenv("OPENSEARCH_PASS")

# AWS Configuration
REGION = "us-east-2"
BUCKET_NAME = "meli-recsys-mvp-iturriago-844339186350-us-east-2-an"
SILVER_S3_KEY = "silver/embeddings/image_embeddings.jsonl"
INDEX_NAME = "products_hybrid_index"

# Initialize AWS Client
s3_client = boto3.client('s3', region_name=REGION)

In [12]:
# ==========================================
# 2. OPENSEARCH CONNECTION
# ==========================================
print("[INFO] Connecting to OpenSearch Cluster...")
client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': 443}],
    http_compress=True,
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=True,
    verify_certs=True,
    ssl_assert_hostname=False,
    ssl_show_warn=False,
    timeout=60,          # FIX 1: Tell Python to wait up to 60 seconds for a response
    max_retries=3,       # FIX 2: If a connection drops, automatically retry
    retry_on_timeout=True # FIX 3: Don't crash immediately on timeout, retry first
)

if client.ping():
    print("[SUCCESS] Connected to OpenSearch successfully!")
else:
    print("[ERROR] Connection failed. Check credentials or endpoint.")

[INFO] Connecting to OpenSearch Cluster...
[SUCCESS] Connected to OpenSearch successfully!


In [13]:
# ==========================================
# 3. K-NN INDEX MAPPING (The Core ML Architecture)
# ==========================================
# This schema tells OpenSearch how to treat our multimodal vectors
index_body = {
    "settings": {
        "index.knn": True, # Crucial: Enables Vector Search
        "index.knn.algo_param.ef_search": 100, # Controls search recall vs speed
        "number_of_shards": 1,
        "number_of_replicas": 0 # Set to 0 to save costs in MVP
    },
    "mappings": {
        "properties": {
            "article_id": {
                "type": "keyword" # Keyword ensures exact matching for Hybrid Search
            },
            "image_embedding": {
                "type": "knn_vector",
                "dimension": 512, # CLIP output dimension
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil", # Cosine similarity for normalized vectors
                    "engine": "lucene" # Fixed: 'lucene' is the production standard for OpenSearch 3.0+
                }
            }
        }
    }
}

# Create index if it doesn't exist
if not client.indices.exists(index=INDEX_NAME):
    print(f"[INFO] Creating k-NN index '{INDEX_NAME}'...")
    client.indices.create(index=INDEX_NAME, body=index_body)
    print(f"[SUCCESS] Index created.")
else:
    print(f"[INFO] Index '{INDEX_NAME}' already exists.")

[INFO] Index 'products_hybrid_index' already exists.


In [14]:
# ==========================================
# 4. BULK INGESTION STREAMING FROM S3
# ==========================================
def generate_actions_from_s3():
    """
    Generator function that streams the JSONL file directly from S3.
    This avoids loading the 50MB file into memory, demonstrating Senior MLOps practices.
    """
    try:
        response = s3_client.get_object(Bucket=BUCKET_NAME, Key=SILVER_S3_KEY)
        # Read the stream line by line
        for line in response['Body'].iter_lines():
            if line:
                doc = json.loads(line.decode('utf-8'))
                
                # Format required by OpenSearch Bulk API
                yield {
                    "_index": INDEX_NAME,
                    "_id": doc["article_id"], # Set article_id as the primary document ID
                    "_source": doc
                }
    except Exception as e:
        print(f"[ERROR] Failed to stream from S3: {e}")

print("[INFO] Resuming Bulk Ingestion with micro-batches...")
try:
    # Use the helper to automatically chunk and parallelize the upload
    success, failed = helpers.bulk(
        client, 
        generate_actions_from_s3(), 
        chunk_size=100,       # FIX 4: Reduced from 500 to 100 to avoid choking the t3.small CPU
        request_timeout=60    # Explicit timeout for the bulk operation
    )
    print(f"\n[SUCCESS] Bulk ingestion complete!")
    print(f"   -> Successfully indexed/updated: {success} documents.")
    if failed:
        print(f"   -> Failed to index: {len(failed)} documents.")
        
except Exception as e:
    print(f"\n[ERROR] Bulk operation failed: {e}")

[INFO] Resuming Bulk Ingestion with micro-batches...

[SUCCESS] Bulk ingestion complete!
   -> Successfully indexed/updated: 5000 documents.


In [15]:
# ==========================================
# 5. DOCUMENT COUNT AND SAMPLE RETRIEVAL
# ==========================================

# Verify the total document count
count_response = client.count(index=INDEX_NAME)
print(f"Total documents in OpenSearch: {count_response['count']}")

# Retrieve a sample document (brute force without vectors for now, just to inspect the structure)
sample_query = {
    "query": {
        "match_all": {}
    },
    "size": 1
}

test_response = client.search(index=INDEX_NAME, body=sample_query)
print("\nSample Document Stored:")
print(test_response['hits']['hits'][0]['_source']['article_id'])
# Do not print the full vector to avoid flooding the screen
print(f"Embedding length: {len(test_response['hits']['hits'][0]['_source']['image_embedding'])} dimensions")

Total documents in OpenSearch: 5000

Sample Document Stored:
0370594006
Embedding length: 512 dimensions
